This notebooks investigates the (validation) results of the GNN models developed using the custom CV approach.   
The results from all validation folds are combined and analyzed.

**It is important to note that these are results are from multiple GNN models!!**   
**All plots/visualisations are based on validation data!!**

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import sim_ranking as sr
import ml_tools as mlt

In [ ]:
wdata = "/Users/claudy/dev/work/data"
results_dir = "/Users/claudy/dev/work/data/sim_ranking/results/gnn/0225_1138_cv_v4p2NZGMDB_empTest_pSAOnly"

In [ ]:
# Injected Parameters
results_dir = "/home/claudy/dev/work/data/sim_ranking/results/gnn/0306_1947_cv_v4p2NZGMDB_v2p3_4e8s_fixedZ1p0Issue"
wdata = "/home/claudy/dev/work/data"


In [ ]:
wdata = Path(wdata)
results_dir = Path(results_dir)
warnings.simplefilter(action='ignore', category=FutureWarning)

**Result Directory:** `{python} str(results_dir)`

In [ ]:
# Load observed data 
run_config = sr.ml.gnn_gm.RunConfig.from_yaml(results_dir / "run_config.yaml")
nzgmdb_ffp = wdata / run_config.rel_obs_data_ffp
obs_data = sr.data.load_obs_nzgmdb(nzgmdb_ffp)

In [ ]:
# Load the combined validation results
val_results = pd.read_parquet(results_dir / "val_results.parquet")

# Load the validation results for each validation fold
cv_dirs = [cur_result for cur_result in results_dir.glob("cv_*") if cur_result.is_dir()]
cv_val_results = {cur_dir.stem: pd.read_parquet(cur_dir / "val_results.parquet") for cur_dir in cv_dirs}
cv_val_res_dfs = {cur_key: sr.ml.gnn_gm.get_residuals(cur_result, ims=run_config.ims) for cur_key, cur_result in cv_val_results.items()}
cv_train_results = {cur_dir.stem: pd.read_parquet(cur_dir / "train_results.parquet") for cur_dir in cv_dirs}

In [ ]:
# Load metrics
metrics = pd.read_pickle(results_dir / "metrics.pickle")

In [ ]:
# Compute residuals
val_res_df = sr.ml.gnn_gm.get_residuals(val_results, ims=run_config.ims)
assert val_res_df.index.equals(val_results.index)
val_res_df["cv_iter"] = val_results["cv_iter"]

# Compute bias and residual standard deviation
val_res_mean_std_df = pd.concat((val_res_df[run_config.ims].mean(axis=0), val_res_df[run_config.ims].std(axis=0)), axis=1)
val_res_mean_std_df.columns = ["mean", "std"]

In [ ]:
# Compute average and standard deviation of predicted standard deviations as function of period
val_pred_std_mean_std_df = pd.concat((val_results[run_config.pred_std_keys].mean(axis=0), val_results[run_config.pred_std_keys].std(axis=0)), axis=1)
val_pred_std_mean_std_df.columns = ["mean", "std"]
val_pred_std_mean_std_df.index = run_config.ims

In [ ]:
# Distance matrix
dist_matrix = sr.utils.calculate_distance_matrix(obs_data.sites, obs_data.site_df)

# Correlation Coefficients
corr_data = sr.LBSiteCorrelationData.from_dist_matrix(dist_matrix, run_config.pSA_ims)

In [ ]:
# Load attention coefficients
# val_attn_coefs = pd.read_parquet(results_dir / "val_attn_coeffs.parquet")

# Add site-to-site distance
# row_ind = dist_matrix.index.get_indexer_for(val_attn_coefs.site_int)
# col_ind = dist_matrix.columns.get_indexer_for(val_attn_coefs.obs_site)
# val_attn_coefs["dist"] = dist_matrix.values[row_ind, col_ind]

In [ ]:
# Plot config
plot_ims = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

if run_config.im_set == sr.constants.IMSet.all:
    plot_ims.append("PGA")
    plot_ims.append("CAV")

## CV & Training Metrics

### CV Overview

In [ ]:
cv_val_n_scenarios = pd.DataFrame({cur_key: [cur_cv_res.shape[0], cur_key, "val"] for cur_key, cur_cv_res in cv_val_results.items()}, index=["n_scenarios", "cv_iter", "type"]).T
cv_train_n_scenarios = pd.DataFrame({cur_key: [cur_cv_res.shape[0], cur_key, "train"] for cur_key, cur_cv_res in cv_train_results.items()}, index=["n_scenarios", "cv_iter", "type"]).T

In [ ]:
n_scenarios_df = pd.concat((cv_val_n_scenarios, cv_train_n_scenarios), axis=0).sort_values("cv_iter").reset_index(drop=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=n_scenarios_df[n_scenarios_df["type"] == "train"], x="cv_iter", y="n_scenarios", color="blue", ax=ax1)
ax1.set_title("Number of Training Scenarios")

sns.barplot(data=n_scenarios_df[n_scenarios_df["type"] == "val"], x="cv_iter", y="n_scenarios", color="red", ax=ax2)
ax2.set_title("Number of Validation Scenarios")

fig.tight_layout()

### Metric Comparison Between CV-Iterations

In [ ]:
# cur_metric_key = "w_loss_hist"

# # Get current metric values
# cur_train_metric_df = metrics.sel[:, :, cur_metric_key + "_train"]
# cur_val_metric_df = metrics.sel[:, :, cur_metric_key + "_val"]

# # Compute mean and std
# cur_mean = cur_train_metric_df.mean(axis=1)
# cur_std = cur_train_metric_df.std(axis=1)

# cur_val_mean = cur_val_metric_df.mean(axis=1)
# cur_val_std = cur_val_metric_df.std(axis=1)


# y_center = (cur_mean.iloc[-10:].mean() + cur_val_mean.iloc[-10:].mean()) / 2
# y_width = max(cur_std.iloc[-10:].mean(), cur_val_std.iloc[-10:].mean()) * 3

In [ ]:
metric_keys = ["w_loss_hist", "loss_hist", "mse_hist", "mean_sigma_hist"]

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys) * 2, 2, -1, (8, 6))

for ix, cur_metric_key in enumerate(metric_keys):
	ax1, ax2 = axs[ix * 2], axs[ix * 2 + 1]
    
    # Get current metric values
	cur_train_metric_df = metrics.sel[:, :, cur_metric_key + "_train"]
	cur_val_metric_df = metrics.sel[:, :, cur_metric_key + "_val"]
    
    # Compute mean and std
	cur_mean = cur_train_metric_df.mean(axis=1)
	cur_std = cur_train_metric_df.std(axis=1)
	
	cur_val_mean = cur_val_metric_df.mean(axis=1)
	cur_val_std = cur_val_metric_df.std(axis=1)
    
    # Compute y-axis limits
	y_center = (cur_mean.iloc[-10:].mean() + cur_val_mean.iloc[-10:].mean()) / 2
	y_width = max(cur_std.iloc[-10:].mean(), cur_val_std.iloc[-10:].mean()) * 3
	y_min = y_center - y_width
	y_max = y_center + y_width
    
    # Plot
	ax1.plot(cur_train_metric_df.index.values, cur_train_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax1.plot(cur_mean.index.values, cur_mean, c="black", linewidth=2.0)
	ax1.set_ylim(y_min, y_max)
	ax1.set_title(f"Training {cur_metric_key}")
	ax1.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
	ax2.plot(cur_val_metric_df.index.values, cur_val_metric_df, c="gray", linewidth=1.0, linestyle="--")
	ax2.plot(cur_val_mean.index.values, cur_val_mean, c="black", linewidth=2.0)
	ax2.set_ylim(y_min, y_max)
	ax2.set_title(f"Validation {cur_metric_key}")
	ax2.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    
fig.tight_layout()

### Comparison Of Median Metrics

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(metric_keys), 1, -1, (10, 6))

for cur_ax, cur_metric_key in zip(axs, metric_keys):
    # Get current metric values
	cur_train_metric_df = metrics.sel[:, :, cur_metric_key + "_train"]
	cur_val_metric_df = metrics.sel[:, :, cur_metric_key + "_val"]
    
    # Get the median and 16th/84th percentiles
	train_median = cur_train_metric_df.median(axis=1)
	train_16, train_84 = np.percentile(cur_train_metric_df, [16, 84], axis=1)
	
	val_median = cur_val_metric_df.median(axis=1)
	val_16, val_84 = np.percentile(cur_val_metric_df, [16, 84], axis=1)
    
    # Work out the y-axis limits
	y_center = (cur_train_metric_df.mean(axis=1).iloc[-10:].mean() + cur_val_metric_df.mean(axis=1).iloc[-10:].mean()) / 2
	y_width = max(cur_train_metric_df.std(axis=1).iloc[-10:].mean() * 2, cur_val_metric_df.std(axis=1).iloc[-10:].mean() * 2)
	y_min = y_center - y_width
	y_max = y_center + y_width

	# Plot
	cur_ax.plot(cur_train_metric_df.index.values, train_median, label="Train")
	cur_ax.fill_between(cur_train_metric_df.index.values, train_16, train_84, alpha=0.5)
	
	cur_ax.plot(cur_val_metric_df.index.values, val_median, label="Validation")
	cur_ax.fill_between(cur_val_metric_df.index.values, val_16, val_84, alpha=0.5)
	
	cur_ax.set_ylim(y_min, y_max)
	cur_ax.set_xlim(cur_train_metric_df.index.values.min(), cur_train_metric_df.index.values.max())
	cur_ax.set_xlabel("Epoch")
	cur_ax.set_ylabel(f"{cur_metric_key}")
	cur_ax.set_title(f"{cur_metric_key}")
	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	cur_ax.legend()
	
	fig.tight_layout()
    

## Residuals

### Residual Histograms

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for cur_im, cur_ax in zip(plot_ims, axs):
    cur_mean = val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = val_res_mean_std_df.loc[cur_im, "std"]
    cur_ax.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    cur_ax.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    cur_ax.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(val_res_df, x=cur_im, bins=bins, ax=cur_ax, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    cur_ax.set_xlabel(f"Residual")
    cur_ax.set_ylabel(f"Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(x_min, x_max )
    cur_ax.set_title(f"{cur_im} Residuals")

    cur_ax.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=cur_ax.transAxes)

    cur_ax.text(0.98, 0.98, f"N = {np.count_nonzero(~val_res_df[cur_im].isna())}", horizontalalignment="right",
                verticalalignment="top", transform=cur_ax.transAxes)
    
fig.tight_layout()

### Bias & (Residual) Standard Deviation

In [ ]:
fig, ax1, ax2, ax3, ax4 =  sr.plot_utils.get_bias_residual_fig()

cv_val_res_mean_values = pd.concat([cur_cv_val_res[run_config.ims].mean().to_frame(cv_key).T for cv_key, cur_cv_val_res in cv_val_res_dfs.items()], axis=0)
cv_val_bias_std = cv_val_res_mean_values.std(axis=0)

# Individual CV biases
ax1.semilogx(np.asarray(run_config.pSA_periods)[:, None], cv_val_res_mean_values[run_config.pSA_ims].T, c="gray", alpha=0.5, linestyle="--", linewidth=1.0)

# Bias - pSA
ax1.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[run_config.pSA_ims, "mean"] - cv_val_bias_std.loc[run_config.pSA_ims], val_res_mean_std_df.loc[run_config.pSA_ims, "mean"] + cv_val_bias_std.loc[run_config.pSA_ims], alpha=0.2, color="blue")

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=cv_val_bias_std.loc[run_config.non_pSA_ims], fmt="o", ms=5)
    ax2.xaxis.set_tick_params(rotation=90)

    # Individual
    for _, cur_row in cv_val_res_mean_values.iterrows():
        ax2.scatter(run_config.non_pSA_ims, cur_row.loc[run_config.non_pSA_ims], c="gray",
                    alpha=0.5, marker="x", s=2.5)

# cv_val_res_std_values = np.stack([cur_cv_val_res[sr.constants.PSA_KEYS].std().values for cur_cv_val_res in cv_val_res_dfs.values()], axis=0)
#
cv_val_res_std_values = pd.concat([cur_cv_val_res[run_config.ims].std().to_frame(cv_key).T for cv_key, cur_cv_val_res in cv_val_res_dfs.items()], axis=0)
cv_val_res_std_std = cv_val_res_std_values.std(axis=0)

# Individual CV residual standard deviations
ax3.semilogx(np.asarray(sr.constants.PERIODS)[:, None], cv_val_res_std_values[run_config.pSA_ims].T, c="gray", alpha=0.5, linestyle="--",linewidth=1.0)

# Residual Standard Deviation - pSA
ax3.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
ax3.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std.loc[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std.loc[run_config.pSA_ims], alpha=0.2, color="blue")
ax3.set_ylim(0.0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
    ax4.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=cv_val_res_std_std.loc[run_config.non_pSA_ims], fmt="o", ms=5)
    ax4.xaxis.set_tick_params(rotation=90)
    ax4.set_ylim(ax3.get_ylim())

    # Individual
    for _, cur_row in cv_val_res_std_values.iterrows():
            ax4.scatter(run_config.non_pSA_ims, cur_row.loc[run_config.non_pSA_ims], c="gray",
                        alpha=0.5, marker="x", s=2.5)

## Predicted Standard Deviation

### Predicted Standard Deviation Histograms

In [ ]:
n_bins = 50
bin_min, bin_max = 0.0, 1.5
bins = np.linspace(bin_min, bin_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))
for cur_im, cur_ax in zip(plot_ims, axs):
	cur_col_key = f"{cur_im}_pred_std"
	# bins = np.linspace(0, max(np.quantile(cur_train_values, 0.98), np.quantile(cur_val_df, 0.98)), n_bins)

	cur_mean = val_results[cur_col_key].mean()
	cur_std = val_results[cur_col_key].std()
	cur_ax.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	cur_ax.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	cur_ax.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
	# cur_ax.hist(cur_train_values, bins=bins, label="Train Data")
    
	sns.histplot(val_results, x=cur_col_key, bins=bins, ax=cur_ax, common_bins=True, hue="cv_iter", multiple="stack", legend=False, palette="viridis")
    
	cur_ax.set_xlabel(f"Predicted Standard Deviation")
	cur_ax.set_ylabel(f"Count")
	cur_ax.set_xlim(bin_min, bin_max)
	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	cur_ax.set_title(f"{cur_im}")

	cur_ax.text(0.98, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2f}$\n"
                         f"min = {val_results[cur_col_key].min():.2f}, max = {val_results[cur_col_key].max():.2f}", horizontalalignment="right", verticalalignment="top", transform=cur_ax.transAxes)
    
fig.tight_layout()

### Predicted Standard Deviation

In [ ]:
cv_val_pred_std_mean = pd.concat([cur_cv_val_result[run_config.pred_std_keys].mean().to_frame(cv_key).T for cv_key, cur_cv_val_result in cv_val_results.items()], axis=0)

fig, ax1, ax2 = sr.plot_utils.get_single_pSA_otherIMs_fig(figsize=(16, 6))
# fig, ax = plt.subplots(1, 1, figsize=(16, 6))

ax1.plot(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Mean", marker=".")
ax1.fill_between(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2)

# Individual CV residual standard deviations
ax1.semilogx(np.asarray(sr.constants.PERIODS)[:, None], cv_val_pred_std_mean[run_config.pred_std_keys_pSA].T, c="gray", alpha=0.5, linestyle="--",linewidth=1.0, label=["Individual CV folds (Mean)"] + [None] * (cv_val_pred_std_mean.shape[0] - 1))

ax1.legend()
ax1.set_ylim(0.0, 1.5)

if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "std"], fmt="o", ms=5)

    # Individual
    for _, cur_row in cv_val_pred_std_mean.iterrows():
        ax2.scatter(run_config.non_pSA_ims, cur_row.loc[run_config.pred_std_keys_non_pSA], c="gray",
                    alpha=0.5, marker="x", s=2.5)
    ax2.set_ylim(ax1.get_ylim())

fig.suptitle("Predicted Standard Deviation")


### Predicted Standard Deviation vs. Residual Standard Deviation

In [ ]:
fig, ax1, ax2 = sr.plot_utils.get_single_pSA_otherIMs_fig(figsize=(16, 6))

ax1.plot(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Predicted Standard Deviation", marker=".")
ax1.fill_between(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2)

ax1.plot(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b", label="Residual Standard Deviation")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")

ax1.legend()
ax1.set_ylim(0.0, 1.5)

if run_config.im_set == sr.constants.IMSet.all:
    ax2.errorbar(run_config.non_pSA_ims, val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "std"], fmt="o", ms=5)
    ax2.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=cv_val_res_std_std[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
    ax2.set_ylim(ax1.get_ylim())
    ax2.xaxis.set_tick_params(rotation=90)

## Trends - Magnitude

In [ ]:
val_res_df["mag"] = obs_data.event_df.loc[val_res_df.event_id, "mag"].values
val_results["mag"] = obs_data.event_df.loc[val_results.event_id, "mag"].values

In [ ]:
# Magnitude bins
mag_bins = [3, 4.5, 6, 8]
assert val_res_df.mag.min() >= mag_bins[0] and val_res_df.mag.max() <= mag_bins[-1]
mag_bin_keys = [f"{mag_bins[i]}_{mag_bins[i+1]}" for i in range(len(mag_bins) - 1)]
val_res_df["mag_bin"] = pd.cut(val_res_df.mag, bins=mag_bins, labels=mag_bin_keys)
val_results["mag_bin"] = pd.cut(val_results.mag, bins=mag_bins, labels=mag_bin_keys)
res_mag_groups = val_res_df.groupby("mag_bin")

### Magnitude Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.histplot(val_res_df.mag, bins=25, ax=ax)
ax.set_title("Magnitude Distribution")
ax.set_xlabel("Magnitude")

for i, (cur_bin, cur_key) in enumerate(zip(mag_bins, mag_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((mag_bins[i] + mag_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={res_mag_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")
   
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(3, 8)
fig.tight_layout()

### Magnitude Residual Histograms

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for i, (cur_im, cur_ax) in enumerate(zip(plot_ims, axs)):
    cur_mean = val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = val_res_mean_std_df.loc[cur_im, "std"]
    cur_ax.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    cur_ax.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    cur_ax.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(val_res_df, x=cur_im, bins=bins, ax=cur_ax, common_bins=True, hue="mag_bin", multiple="stack", legend=True if i==0 else False, palette="viridis", hue_order=list(reversed(mag_bin_keys)))
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    cur_ax.set_xlabel(f"Residual")
    cur_ax.set_ylabel(f"Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(x_min, x_max )
    cur_ax.set_title(f"{cur_im} Residuals")

    cur_ax.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=cur_ax.transAxes)
    
    cur_ax.text(0.98, 0.98, f"N = {np.count_nonzero(~val_res_df[cur_im].isna())}", horizontalalignment="right",
                verticalalignment="top", transform=cur_ax.transAxes)
    
fig.tight_layout()

### Magnitude Bias & (Residual) Standard Deviation

In [ ]:
res_mag_bias = res_mag_groups[run_config.ims].mean()
res_mag_bias_std = res_mag_groups[run_config.ims].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(mag_bin_keys))))) 

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias
for i, (cur_key, cur_group) in enumerate(res_mag_groups):
	ax1.semilogx(sr.constants.PERIODS, res_mag_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {res_mag_groups.size()[cur_key]}", c=colors[i])
    
ax1.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cv_val_bias_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")
ax1.legend()

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=cv_val_bias_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax2.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_mag_groups):
			ax2.scatter(run_config.non_pSA_ims, res_mag_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation
for i, (cur_key, cur_group) in enumerate(res_mag_groups):
	ax3.semilogx(sr.constants.PERIODS, res_mag_bias_std.loc[cur_key, sr.constants.PSA_KEYS], label=cur_key, c=colors[i])
    
ax3.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
ax3.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")
ax3.set_ylim(0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax4.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=cv_val_res_std_std.loc[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax4.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_mag_groups):
		ax4.scatter(run_config.non_pSA_ims, res_mag_bias_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
	ax4.set_ylim(ax3.get_ylim())

### Magnitude - Predicted Standard Deviation

In [ ]:
colors = list(reversed(cm.viridis(np.linspace(0, 1, len(mag_bin_keys))))) 

# fig, ax = plt.subplots(1, 1, figsize=(16, 6))

fig, ax1, ax2 = sr.plot_utils.get_single_pSA_otherIMs_fig(figsize=(16, 6))

# Overall
ax1.plot(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Mean", marker=".")
ax1.fill_between(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2)

# Individual CV residual standard deviations
for i, cur_key in enumerate(mag_bin_keys):
	cur_pred_std_mean = val_results[val_results.mag_bin == cur_key][run_config.pred_std_keys_pSA].mean()
	ax1.semilogx(sr.constants.PERIODS, cur_pred_std_mean, label=f"{cur_key} - N = {res_mag_groups.size()[cur_key]}", c=colors[i])
    
ax1.legend()
ax1.set_ylim(0.0, 1.5)

if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "std"], fmt="o", ms=5)
	ax2.xaxis.set_tick_params(rotation=90)
	for i, cur_key in enumerate(mag_bin_keys):
		cur_pred_std_mean = val_results[val_results.mag_bin == cur_key][run_config.pred_std_keys_non_pSA].mean()
		ax2.scatter(run_config.non_pSA_ims, cur_pred_std_mean, color=colors[i], label=f"{cur_key} - N = {res_mag_groups.size()[cur_key]}")
	ax2.set_ylim(ax1.get_ylim())

## Trends - $R_{rup}$

In [ ]:
val_res_df["rrup"] = obs_data.record_df.loc[val_res_df.index, "rrup"].values
val_results["rrup"] = obs_data.record_df.loc[val_results.index, "rrup"].values

In [ ]:
# Distance bins
dist_bins = [0, 25, 50, 100, 225]
assert val_res_df.rrup.min() >= dist_bins[0] and val_res_df.rrup.max() <= dist_bins[-1]
dist_bin_keys = [f"{dist_bins[i]}_{dist_bins[i+1]}" for i in range(len(dist_bins) - 1)]
val_res_df["dist_bin"] = pd.cut(val_res_df.rrup, bins=dist_bins, labels=dist_bin_keys)
val_results["dist_bin"] = pd.cut(val_results.rrup, bins=dist_bins, labels=dist_bin_keys)
res_dist_groups = val_res_df.groupby("dist_bin")

### $R_{Rup}$ Distribution

In [ ]:
# Distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
sns.histplot(val_res_df.rrup, bins=25, ax=ax)
ax.set_title("$R_{Rup}$ Distribution")
ax.set_xlabel("$R_{Rup}$")

for i, (cur_bin, cur_key) in enumerate(zip(dist_bins, dist_bin_keys)):
	ax.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax.text((dist_bins[i] + dist_bins[i+1]) / 2, ax.get_ylim()[1] * 0.98, f"N={res_dist_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")
    
ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax.set_xlim(0, 200)
fig.tight_layout()

### $R_{Rup}$ Residual Histograms

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for i, (cur_im, cur_ax) in enumerate(zip(plot_ims, axs)):
    cur_mean = val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = val_res_mean_std_df.loc[cur_im, "std"]
    cur_ax.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    cur_ax.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    cur_ax.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(val_res_df, x=cur_im, bins=bins, ax=cur_ax, common_bins=True, hue="dist_bin", multiple="stack", legend=True if i==0 else False, palette="viridis", hue_order=list(reversed(dist_bin_keys)))
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    cur_ax.set_xlabel(f"Residual")
    cur_ax.set_ylabel(f"Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(x_min, x_max )
    cur_ax.set_title(f"{cur_im} Residuals")

    cur_ax.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=cur_ax.transAxes)
    
    cur_ax.text(0.98, 0.98, f"N = {np.count_nonzero(~val_res_df[cur_im].isna())}", horizontalalignment="right",
                verticalalignment="top", transform=cur_ax.transAxes)
    
fig.tight_layout()

### $R_{Rup}$ Bias & (Residual) Standard Deviation

In [ ]:
res_dist_bias = res_dist_groups[run_config.ims].mean()
res_dist_std = res_dist_groups[run_config.ims].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys))))) 

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias
ax1.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cv_val_bias_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")
for i, (cur_key, cur_group) in enumerate(res_dist_groups):
	ax1.semilogx(sr.constants.PERIODS, res_dist_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {res_dist_groups.size()[cur_key]}", c=colors[i])
ax1.legend(title="Distance Bin")

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=cv_val_bias_std[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax2.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_dist_groups):
			ax2.scatter(run_config.non_pSA_ims, res_dist_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation
ax3.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
ax3.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")
for i, (cur_key, cur_group) in enumerate(res_dist_groups):
	ax3.semilogx(sr.constants.PERIODS, res_dist_std.loc[cur_key, sr.constants.PSA_KEYS], label=cur_key, c=colors[i])
ax3.set_ylim(0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax4.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=cv_val_res_std_std[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax4.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_dist_groups):
		ax4.scatter(run_config.non_pSA_ims, res_dist_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
	ax4.set_ylim(ax3.get_ylim())

### $R_{Rup}$ - Predicted Standard Deviation

In [ ]:
colors = list(reversed(cm.viridis(np.linspace(0, 1, len(dist_bin_keys))))) 

fig, ax1, ax2 = sr.plot_utils.get_single_pSA_otherIMs_fig(figsize=(16, 6))

# pSA
ax1.plot(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Mean", marker=".")
ax1.fill_between(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2)
for i, cur_key in enumerate(dist_bin_keys):
	cur_pred_std_mean = val_results[val_results.dist_bin == cur_key][run_config.pred_std_keys_pSA].mean()
	ax1.semilogx(sr.constants.PERIODS, cur_pred_std_mean, label=f"{cur_key} - N = {res_dist_groups.size()[cur_key]}", c=colors[i])
ax1.legend()
ax1.set_ylim(0.0, 1.5)

# Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "std"], fmt="o", ms=5)
	for i, cur_key in enumerate(dist_bin_keys):
		cur_pred_std_mean = val_results[val_results.dist_bin == cur_key][run_config.pred_std_keys_non_pSA].mean()
		ax2.scatter(run_config.non_pSA_ims, cur_pred_std_mean, color=colors[i])
	ax2.set_ylim(ax1.get_ylim())

## Trends - Degree of Constraint

In [ ]:
val_results = sr.utils.compute_degree_of_constraint(val_results, corr_data)
val_res_df["doc"] = val_results["doc"]

### DoC Distribution



In [ ]:
# DoC bins
doc_bins = [0, 1.0, 2.25, 3.75, 8.0]
assert val_res_df.doc.min() >= doc_bins[0] and val_res_df.doc.max() <= doc_bins[-1]
doc_bin_keys = [f"{doc_bins[i]}_{doc_bins[i + 1]}" for i in range(len(doc_bins) - 1)]
val_res_df["doc_bin"] = pd.cut(val_res_df.doc, bins=doc_bins, labels=doc_bin_keys)
val_results["doc_bin"] = pd.cut(val_results.doc, bins=doc_bins, labels=doc_bin_keys)
res_doc_groups = val_res_df.groupby("doc_bin")

In [ ]:
res_doc_groups.size()

In [ ]:
# Distribution
fig, ax1 = plt.subplots(1, 1, figsize=(16, 6))

sns.histplot(val_res_df.doc, bins=40, ax=ax1)
ax1.set_xlabel("Degree of Constraint")

for i, (cur_bin, cur_key) in enumerate(zip(doc_bins, doc_bin_keys)):
	ax1.axvline(cur_bin, color="k", linestyle="--", linewidth=1.0)
	ax1.text((doc_bins[i] + doc_bins[i + 1]) / 2, ax1.get_ylim()[1] * 0.98, f"N={res_doc_groups.size()[cur_key]}", horizontalalignment="center", verticalalignment="top")

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0, max(doc_bins))

ax1.set_title("Degree of Constraint Distribution")
fig.tight_layout()

### Degree of Constraint Residual Histograms

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for i, (cur_im, cur_ax) in enumerate(zip(plot_ims, axs)):
    cur_mean = val_res_mean_std_df.loc[cur_im, "mean"]
    cur_std = val_res_mean_std_df.loc[cur_im, "std"]
    cur_ax.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
    cur_ax.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
    cur_ax.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
    
    sns.histplot(val_res_df, x=cur_im, bins=bins, ax=cur_ax, common_bins=True, hue="doc_bin", multiple="stack", legend=True if i==0 else False, palette="viridis", hue_order=list(reversed(doc_bin_keys)))
    
    # cur_ax.hist(val_res_df[cur_im], bins=bins, label="Train Data")
    cur_ax.set_xlabel(f"Residual")
    cur_ax.set_ylabel(f"Count")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(x_min, x_max )
    cur_ax.set_title(f"{cur_im} Residuals")

    cur_ax.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
                verticalalignment="top", transform=cur_ax.transAxes)
    
    cur_ax.text(0.98, 0.98, f"N = {np.count_nonzero(~val_res_df[cur_im].isna())}", horizontalalignment="right",
                verticalalignment="top", transform=cur_ax.transAxes)
    
    
fig.tight_layout()

### Degree of Constraint Bias & (Residual) Standard Deviation

In [ ]:
res_doc_bias = res_doc_groups[run_config.ims].mean()
res_doc_std = res_doc_groups[run_config.ims].std()

colors = list(reversed(cm.viridis(np.linspace(0, 1, len(doc_bin_keys)))))

fig, ax1, ax2, ax3, ax4 = sr.plot_utils.get_bias_residual_fig(figsize=(16, 6))

# Bias - pSA
ax1.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cv_val_bias_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cv_val_bias_std[run_config.pSA_ims], alpha=0.2, color="blue")
for i, (cur_key, cur_group) in enumerate(res_doc_groups):
	ax1.semilogx(sr.constants.PERIODS, res_doc_bias.loc[cur_key, sr.constants.PSA_KEYS], label=f"{cur_key} - N = {res_doc_groups.size()[cur_key]}", c=colors[i])
ax1.set_title("Bias")
ax1.legend(title="DoC Bins")

# Bias - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=cv_val_bias_std[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax2.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_doc_groups):
		ax2.scatter(run_config.non_pSA_ims, res_doc_bias.loc[cur_key, run_config.non_pSA_ims], color=colors[i])

# Residual Standard Deviation - pSA
ax3.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
ax3.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std[run_config.pSA_ims], val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std[run_config.pSA_ims], alpha=0.2, color="blue")
for i, (cur_key, cur_group) in enumerate(res_doc_groups):
	ax3.semilogx(sr.constants.PERIODS, res_doc_std.loc[cur_key, sr.constants.PSA_KEYS], label=cur_key, c=colors[i])
ax3.set_ylim(0, 1.5)

# Residual Standard Deviation - Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax4.errorbar(run_config.non_pSA_ims, val_res_mean_std_df.loc[run_config.non_pSA_ims, "std"], yerr=cv_val_res_std_std[run_config.non_pSA_ims], fmt="o", ms=5, c="b")
	ax4.xaxis.set_tick_params(rotation=90)
	for i, (cur_key, cur_group) in enumerate(res_doc_groups):
		ax4.scatter(run_config.non_pSA_ims, res_doc_std.loc[cur_key, run_config.non_pSA_ims], color=colors[i])
	ax4.set_ylim(ax3.get_ylim())


### Degree of Constraint - Predicted Standard Deviation

In [ ]:
colors = list(reversed(cm.viridis(np.linspace(0, 1, len(doc_bin_keys)))))

fig, ax1, ax2 = sr.plot_utils.get_single_pSA_otherIMs_fig(figsize=(16, 6))

# pSA
ax1.plot(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], label="Mean", marker=".")
ax1.fill_between(sr.constants.PERIODS, val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + val_pred_std_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], alpha=0.2)
for i, cur_key in enumerate(doc_bin_keys):
	cur_pred_std_mean = val_results[val_results.doc_bin == cur_key][run_config.pred_std_keys_pSA].mean()
	ax1.semilogx(sr.constants.PERIODS, cur_pred_std_mean, label=f"{cur_key} - N = {res_doc_groups.size()[cur_key]}", c=colors[i])
ax1.legend()
ax1.set_ylim(0.0, 1.5)

# Non-pSA
if run_config.im_set == sr.constants.IMSet.all:
	ax2.errorbar(run_config.non_pSA_ims, val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "mean"], yerr=val_pred_std_mean_std_df.loc[run_config.non_pSA_ims, "std"], fmt="o", ms=5)
	for i, cur_key in enumerate(doc_bin_keys):
		cur_pred_std_mean = val_results[val_results.doc_bin == cur_key][run_config.pred_std_keys_non_pSA].mean()
		ax2.scatter(run_config.non_pSA_ims, cur_pred_std_mean, color=colors[i])
	ax2.set_ylim(ax1.get_ylim())

### Residual vs Degree of Constraint (Scatter)

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

mask = val_res_df.doc < 2
for i, (cur_im, cur_ax) in enumerate(zip(plot_ims, axs)):
    # Normal
    cur_ax.scatter(val_res_df.doc, val_res_df[cur_im], alpha=0.1)
    cur_ax.axhline(0, color="k", linestyle="--")
    
    bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(val_res_df.doc.values, val_res_df[cur_im].values, n_points_per_bin=2500, ignore_nans=True)
    # bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(val_res_df.doc.values, val_res_df[cur_im].values, bins=bins, ignore_nans=True)
    
    cur_ax.plot(bin_centers, bin_means, c="r", label="Mean", marker=".")
    cur_ax.plot(bin_centers, bin_means + bin_stds, c="r", linestyle="--", label="Std")
    cur_ax.plot(bin_centers, bin_means - bin_stds, c="r", linestyle="--")
    
    cur_ax.set_ylabel(f"Residual")
    cur_ax.set_xlabel("Degree of Constraint")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_ylim(-2.5, 2.5)
    cur_ax.set_title(f"{cur_im} Residuals")

### Predicted Standard Deviation vs Degree of Constraint (Scatter)

In [ ]:
bins = np.asarray(doc_bins)

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

mask = val_results.doc < 2
for i, (cur_im, cur_ax) in enumerate(zip(plot_ims, axs)):
    # Normal
    cur_ax.scatter(val_results.doc, val_results[f"{cur_im}_pred_std"], alpha=0.1)

    bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(val_res_df.doc.values, val_results[f"{cur_im}_pred_std"].values, n_points_per_bin=2500, ignore_nans=True)
    # bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(val_results.doc.values, val_results[f"{cur_im}_pred_std"].values, bins=bins, ignore_nans=True)
    
    cur_ax.plot(bin_centers, bin_means, c="r", label="Mean", marker=".")
    cur_ax.plot(bin_centers, bin_means + bin_stds, c="r", linestyle="--", label="Std")
    cur_ax.plot(bin_centers, bin_means - bin_stds, c="r", linestyle="--")
    
    cur_ax.set_ylabel(f"Predicted Standard Deviation")
    cur_ax.set_xlabel("Degree of Constraint")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_ylim(0, 1.5)
    cur_ax.set_title(cur_im)

## Trends - Predicted Standard Deviation

### Predicted Standard Deviation vs. Residual (Scatter)

In [ ]:
pred_std_bins = [0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2]

fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))

for cur_im, cur_ax in zip(plot_ims, axs):
    cur_ax.scatter(val_res_df[cur_im].abs(), val_results[f"{cur_im}_pred_std"], alpha=0.1)
    # cur_ax.axhline(0, color="k", linestyle="--")
    
    # bin_centers, bin_means, bin_stds = mlt.utils.compute_binned_trend(val_results[f"{cur_im}_pred_std"].values, val_res_df[cur_im].values, bins=np.asarray(pred_std_bins), ignore_nans=True)
    bin_centers, bin_means, bin_stds = mlt.utils.compute_count_binned_trend(val_res_df[cur_im].abs().values, val_results[f"{cur_im}_pred_std"].values, n_points_per_bin=250, ignore_nans=True)

    cur_ax.plot(bin_centers, bin_means, c="r", label="Mean")
    cur_ax.plot(bin_centers, bin_means + bin_stds, c="r", linestyle="--", label="Std")
    cur_ax.plot(bin_centers, bin_means - bin_stds, c="r", linestyle="--")
    
    cur_ax.set_xlabel(f"Absolute Residual")
    cur_ax.set_ylabel("Predicted Standard Deviation")
    cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    cur_ax.set_xlim(0, 2)
    cur_ax.set_title(cur_im)
    


## Subgroup Analysis

Investigate the performance of the most interesting scenarions, i.e. larger magnitude, short site-to-source distance and somewhat well constrainted.

### Definitions
| Group Name                                   | $M_w$ Range | $R_{Rup}$ Range | $C_s$ Range |
| -------------------------------------------- | ----------- | --------------- |-------------|
| High Importance, Well Constrained (HIWC)     | 5+          | 0, 50           | 2.0+        |
| High Importance, Limited Constraint (HILC)   | 5+          | 0, 50           | 0.0, 2.0    |
| Medium Importance, Well Constraint (MIWC)    | 4.5, 5      | 50, 100         | 2.0+        |
| Medium Importance, Limited Constraint (MILC) | 4.5, 5      | 50, 100         | 0.0, 2.0    |
| Low Importance, Well Constraint (LIWC)       | 3, 4.5      | 100+            | 2.0+        |
| Low Importance, Limited Constraint (LILC)    | 3, 4.5      | 100+            | 0.0, 2.0    |


In [ ]:
# Subgroup Definition
hiwc_mask = (val_results.mag > 5.0) & (val_results.rrup < 50) & (val_results.doc > 2)
val_results.loc[hiwc_mask, "imp_group"] = "HIWC"

hilc_mask = (val_results.mag > 5.0) & (val_results.rrup < 50) & (val_results.doc < 2)
val_results.loc[hilc_mask, "imp_group"] = "HILC"

miwc_mask = (val_results.mag > 4.5) & (val_results.mag < 5.0) & (val_results.rrup > 50) & (val_results.rrup < 100) & (val_results.doc > 2)
val_results.loc[miwc_mask, "imp_group"] = "MIWC"

milc_mask = (val_results.mag > 4.5) & (val_results.mag < 5.0) & (val_results.rrup > 50) & (val_results.rrup < 100) & (val_results.doc < 2)
val_results.loc[milc_mask, "imp_group"] = "MILC"

liwc_mask = (val_results.mag < 4.5) & (val_results.rrup > 100) & (val_results.doc > 2)
val_results.loc[liwc_mask, "imp_group"] = "LIWC"

lilc_mask = (val_results.mag < 4.5) & (val_results.rrup > 100) & (val_results.doc < 2)
val_results.loc[lilc_mask, "imp_group"] = "LILC"

val_res_df["imp_group"] = val_results["imp_group"]

In [ ]:
print(f"Datapoints per group:")
val_results.groupby("imp_group").size()

### Subgroup Distributions

In [ ]:
limited_cs_names = ["HILC", "MILC", "LILC"]
limited_cs_colors = ["coral", "lightgreen", "gray"]

well_const_group_names = ["HIWC", "MIWC", "LIWC"]
well_const_colors = ["darkred", "darkgreen", "black"]


group_names = well_const_group_names + limited_cs_names
group_colors = well_const_colors + limited_cs_colors

# group_names = ["HIWC", "HILC", "MIWC", "MILC", "LIWC", "LILC"]
# group_colors = ["darkred", "red", "darkgreen", "green", "black", "gray"]

In [ ]:
# Degree of Constraint Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(val_res_df, x="doc", bins=25, ax=ax1, hue=val_results.imp_group, multiple="stack",
             palette=group_colors, hue_order=group_names)
ax1.set_xlabel("Degree of Constraint")

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0, None)

sns.histplot(val_res_df.loc[val_res_df.doc < 2.0], x="doc", bins=25, ax=ax2,
             hue=val_results.loc[val_res_df.doc < 2.0, "imp_group"], multiple="stack",
             palette=group_colors, hue_order=group_names)
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xlim(0, None)
ax2.set_xlabel("Degree of Constraint")

fig.suptitle("Degree of Constraint Distribution")
fig.tight_layout()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))


# Rrup Distribution
sns.histplot(val_res_df, x="rrup", bins=25, ax=ax1, hue=val_results.imp_group, multiple="stack",
             palette=group_colors, hue_order=group_names)
ax1.set_title("$R_{Rup}$ Distribution")
ax1.set_xlabel("$R_{Rup}$")

ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0, 200)

# Magnitude
sns.histplot(val_res_df, x="mag", bins=25, ax=ax2, hue=val_results.imp_group, multiple="stack",
             palette=group_colors, hue_order=group_names)
ax2.set_title("Magnitude Distribution")
ax2.set_xlabel("Magnitude")

ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xlim(3, 8)

fig.tight_layout()

### Residual Histograms

In [ ]:
# cur_im = "pSA_0.1"
n_bins = 50
x_min, x_max = -2.5, 2.5
bins = np.linspace(x_min, x_max, n_bins)

# fig, axs = mlt.plotting.get_fig_axes(len(plot_ims), 2, -1, (8, 6))
fig, axs = mlt.plotting.get_fig_axes(2 * len(plot_ims), 2, -1, (8, 6))

for i, cur_im in enumerate(plot_ims):
	ax1, ax2 = axs[i * 2], axs[i * 2 + 1]
	
	# Histogram
	cur_mean = val_res_mean_std_df.loc[cur_im, "mean"]
	cur_std = val_res_mean_std_df.loc[cur_im, "std"]
	ax1.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax1.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax1.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
	sns.histplot(val_res_df, x=cur_im, bins=bins, ax=ax1, common_bins=True, hue="imp_group", multiple="stack", legend=True if i==0 else False, hue_order=group_names, palette=group_colors)
	
	ax1.set_xlabel(f"Residual")
	ax1.set_ylabel(f"Count")
	ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax1.set_xlim(x_min, x_max )
	ax1.set_title(f"{cur_im} Residuals")
	
	ax1.text(0.02, 0.98, f"$\mu = {cur_mean:.2f}$, $\sigma = {cur_std:.2}$", horizontalalignment="left",
	            verticalalignment="top", transform=ax1.transAxes)
	
	# KDE
	sns.kdeplot(val_res_df.loc[val_res_df.imp_group.isin(limited_cs_names)], x=cur_im, ax=ax2, hue="imp_group",
	            hue_order=limited_cs_names, palette=limited_cs_colors, common_norm=False, legend=False, linestyle= "--")
	sns.kdeplot(val_res_df.loc[val_res_df.imp_group.isin(well_const_group_names)], x=cur_im, ax=ax2, hue="imp_group",
	            hue_order=well_const_group_names, palette=well_const_colors, common_norm=False, legend=False, linestyle= "-")
	
	ax2.axvline(cur_mean, color="k", linestyle="-", label="Mean", linewidth=1.5)
	ax2.axvline(cur_mean + cur_std, color="k", linestyle="--", label="Std", linewidth=1.0)
	ax2.axvline(cur_mean - cur_std, color="k", linestyle="--", linewidth=1.0)
	
	ax2.set_xlabel(f"Residual")
	ax2.set_ylabel(f"Density")
	ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
	ax2.set_xlim(x_min, x_max )
	ax2.set_title(f"{cur_im} Residuals")
	
	ax2.text(0.98, 0.98, f"N = {np.count_nonzero(~val_res_df[cur_im].isna())}", horizontalalignment="right",
	            verticalalignment="top", transform=ax2.transAxes)

    
fig.tight_layout()

### Bias & Standard Deviation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

cv_val_res_mean_values = np.stack([cur_cv_val_res[sr.constants.PSA_KEYS].mean().values for cur_cv_val_res in cv_val_res_dfs.values()], axis=0)
cv_val_bias_std = cv_val_res_mean_values.std(axis=0)

# Bias
for cur_imp_grp, cur_c in zip(limited_cs_names, limited_cs_colors):
	ax1.semilogx(sr.constants.PERIODS, val_res_df.loc[val_res_df.imp_group == cur_imp_grp, sr.constants.PSA_KEYS].mean(), c= cur_c, label=cur_imp_grp, linestyle="--")
    
for cur_imp_grp, cur_c in zip(well_const_group_names, well_const_colors):
    	ax1.semilogx(sr.constants.PERIODS, val_res_df.loc[val_res_df.imp_group == cur_imp_grp, sr.constants.PSA_KEYS].mean(), c= cur_c, label=cur_imp_grp)

ax1.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"], c="b", label="Overall - Mean")
ax1.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] - cv_val_bias_std, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "mean"] + cv_val_bias_std, alpha=0.2, color="blue", label="Overall - Std")

ax1.axhline(0, color="k", linestyle="--")
ax1.set_xlabel(f"Period (s)")
ax1.set_ylabel(f"Bias")
ax1.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax1.set_xlim(0.01, 10)
ax1.set_ylim(-0.5, 0.5)
ax1.set_title("Bias")
ax1.legend()

cv_val_res_std_values = np.stack([cur_cv_val_res[sr.constants.PSA_KEYS].std().values for cur_cv_val_res in cv_val_res_dfs.values()], axis=0)
cv_val_res_std_std = cv_val_res_std_values.std(axis=0)

# Residual Standard Deviation
ax2.semilogx(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"], c="b")
ax2.fill_between(sr.constants.PERIODS, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] - cv_val_res_std_std, val_res_mean_std_df.loc[sr.constants.PSA_KEYS, "std"] + cv_val_res_std_std, alpha=0.2, color="blue")

for cur_imp_grp, cur_c in zip(limited_cs_names, limited_cs_colors):
	ax2.semilogx(sr.constants.PERIODS, val_res_df.loc[val_res_df.imp_group == cur_imp_grp, sr.constants.PSA_KEYS].std(), c= cur_c, label=cur_imp_grp, linestyle="--")
    
for cur_imp_grp, cur_c in zip(well_const_group_names, well_const_colors):
    	ax2.semilogx(sr.constants.PERIODS, val_res_df.loc[val_res_df.imp_group == cur_imp_grp, sr.constants.PSA_KEYS].std(), c= cur_c, label=cur_imp_grp)


ax2.set_xlabel(f"Period (s)")
ax2.set_ylabel(f"Standard deviation of residuals")
ax2.grid(linewidth=0.5, alpha=0.5, linestyle="--")
ax2.set_xlim(0.01, 10)
ax2.set_ylim(0, 1.5)
ax2.set_title("Residual Standard Deviation");

fig.tight_layout()

In [ ]:
## Attention Coefficients

In [ ]:
### Distance

In [ ]:
# cv_iter_loss_df = metrics.sel[:, :, "loss_hist_val"].min(axis=0)

In [ ]:
# # Attention coefficient values with respect to distance per CV iteration
# n_cv_iters = len(val_attn_coefs.cv_iter.unique())
# conv_cols = [cur_col for cur_col in val_attn_coefs.columns if cur_col.startswith("conv_")]

# fig, axs = mlt.plotting.get_fig_axes(n_cv_iters, 2, -1, (8, 6))

# for ix, (cur_cv_iter, cur_ax) in enumerate(zip(val_attn_coefs.cv_iter.unique(), axs)):
# 	cur_data = val_attn_coefs.loc[val_attn_coefs.cv_iter == cur_cv_iter]

# 	for cur_conv in conv_cols:
# 	    cur_ax.scatter(cur_data.dist, cur_data[cur_conv], label=cur_conv, s=2.5)

# 	if ix == 0:
# 		cur_ax.legend()

# 	cur_ax.set_xlabel("Site-to-site distance (km)")
# 	cur_ax.set_ylabel("Attention coefficient")
# 	cur_ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
# 	cur_ax.set_title(f"{cur_cv_iter} - Loss: {cv_iter_loss_df[cur_cv_iter]:.3f}")
# 	cur_ax.set_ylim(0, 1)

# fig.tight_layout()